In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [116]:
import pandas as pd
import os

# dir = "/content/drive/MyDrive/HLT project/Project files/sentiment_annotated_with_texts.csv"
dir = "/home/lesi-nedo/Desktop/master/second-semester/HLT/project/sentiment_annotated_with_texts.csv"

with open(dir, 'r') as f:
  sent = pd.read_csv(f, delimiter=',', quotechar='"')



In [117]:
%matplotlib inline


import matplotlib.pyplot as plt

plt.style.use('ggplot')
from itertools import chain
import nltk
import sklearn
import numpy as np
import scipy.stats
import string
import re
import string
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer

! pip install sklearn-crfsuite
import sklearn_crfsuite
from sklearn_crfsuite import scorers
from sklearn_crfsuite import metrics
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
from nltk.stem import WordNetLemmatizer
nltk.download("punkt")
nltk.download('wordnet')




[nltk_data] Downloading package punkt to /home/lesi-nedo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/lesi-nedo/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to /home/lesi-nedo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/lesi-
[nltk_data]     nedo/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [120]:
all_text = sent[['text', 'true_sentiment']].copy()
all_text['text'] = all_text['text']
all_text_c = all_text.copy()
all_text_t = all_text.copy()

In [121]:
from __future__ import absolute_import

from six.moves import zip
from tqdm import tqdm
import pycrfsuite

from sklearn_crfsuite._fileresource import FileResource
from sklearn_crfsuite.trainer import LinePerIterationTrainer
from sklearn_crfsuite.compat import BaseEstimator


class CRF(BaseEstimator):
    """
    python-crfsuite wrapper with interface siimlar to scikit-learn.
    It allows to use a familiar fit/predict interface and scikit-learn
    model selection utilities (cross-validation, hyperparameter optimization).

    Unlike pycrfsuite.Trainer / pycrfsuite.Tagger this object is picklable;
    on-disk files are managed automatically.

    Parameters
    ----------
    algorithm : str, optional (default='lbfgs')
        Training algorithm. Allowed values:

        * ``'lbfgs'`` - Gradient descent using the L-BFGS method
        * ``'l2sgd'`` - Stochastic Gradient Descent with L2 regularization term
        * ``'ap'`` - Averaged Perceptron
        * ``'pa'`` - Passive Aggressive (PA)
        * ``'arow'`` - Adaptive Regularization Of Weight Vector (AROW)

    min_freq : float, optional (default=0)
        Cut-off threshold for occurrence
        frequency of a feature. CRFsuite will ignore features whose
        frequencies of occurrences in the training data are no greater
        than `min_freq`. The default is no cut-off.

    all_possible_states : bool, optional (default=False)
        Specify whether CRFsuite generates state features that do not even
        occur in the training data (i.e., negative state features).
        When True, CRFsuite generates state features that associate all of
        possible combinations between attributes and labels.

        Suppose that the numbers of attributes and labels are A and L
        respectively, this function will generate (A * L) features.
        Enabling this function may improve the labeling accuracy because
        the CRF model can learn the condition where an item is not predicted
        to its reference label. However, this function may also increase
        the number of features and slow down the training process
        drastically. This function is disabled by default.

    all_possible_transitions : bool, optional (default=False)
        Specify whether CRFsuite generates transition features that
        do not even occur in the training data (i.e., negative transition
        features). When True, CRFsuite generates transition features that
        associate all of possible label pairs. Suppose that the number
        of labels in the training data is L, this function will
        generate (L * L) transition features.
        This function is disabled by default.

    c1 : float, optional (default=0)
        The coefficient for L1 regularization.
        If a non-zero value is specified, CRFsuite switches to the
        Orthant-Wise Limited-memory Quasi-Newton (OWL-QN) method.
        The default value is zero (no L1 regularization).

        Supported training algorithms: lbfgs

    c2 : float, optional (default=1.0)
        The coefficient for L2 regularization.

        Supported training algorithms: l2sgd, lbfgs

    max_iterations : int, optional (default=None)
        The maximum number of iterations for optimization algorithms.
        Default value depends on training algorithm:

        * lbfgs - unlimited;
        * l2sgd - 1000;
        * ap - 100;
        * pa - 100;
        * arow - 100.

    num_memories : int, optional (default=6)
        The number of limited memories for approximating the inverse hessian
        matrix.

        Supported training algorithms: lbfgs

    epsilon : float, optional (default=1e-5)
        The epsilon parameter that determines the condition of convergence.

        Supported training algorithms: ap, arow, lbfgs, pa

    period : int, optional (default=10)
        The duration of iterations to test the stopping criterion.

        Supported training algorithms: l2sgd, lbfgs

    delta : float, optional (default=1e-5)
        The threshold for the stopping criterion; an iteration stops
        when the improvement of the log likelihood over the last
        `period` iterations is no greater than this threshold.

        Supported training algorithms: l2sgd, lbfgs

    linesearch : str, optional (default='MoreThuente')
        The line search algorithm used in L-BFGS updates. Allowed values:

        * ``'MoreThuente'`` - More and Thuente's method;
        * ``'Backtracking'`` - backtracking method with regular Wolfe condition;
        * ``'StrongBacktracking'`` -  backtracking method with strong Wolfe
          condition.

        Supported training algorithms: lbfgs

    max_linesearch : int, optional (default=20)
        The maximum number of trials for the line search algorithm.

        Supported training algorithms: lbfgs

    calibration_eta : float, optional (default=0.1)
        The initial value of learning rate (eta) used for calibration.

        Supported training algorithms: l2sgd

    calibration_rate : float, optional (default=2.0)
        The rate of increase/decrease of learning rate for calibration.

        Supported training algorithms: l2sgd

    calibration_samples : int, optional (default=1000)
        The number of instances used for calibration.
        The calibration routine randomly chooses instances no larger
        than `calibration_samples`.

        Supported training algorithms: l2sgd

    calibration_candidates : int, optional (default=10)
        The number of candidates of learning rate.
        The calibration routine terminates after finding
        `calibration_samples` candidates of learning rates
        that can increase log-likelihood.

        Supported training algorithms: l2sgd

    calibration_max_trials : int, optional (default=20)
        The maximum number of trials of learning rates for calibration.
        The calibration routine terminates after trying
        `calibration_max_trials` candidate values of learning rates.

        Supported training algorithms: l2sgd

    pa_type : int, optional (default=1)
        The strategy for updating feature weights. Allowed values:

        * 0 - PA without slack variables;
        * 1 - PA type I;
        * 2 - PA type II.

        Supported training algorithms: pa

    c : float, optional (default=1)
        Aggressiveness parameter (used only for PA-I and PA-II).
        This parameter controls the influence of the slack term on the
        objective function.

        Supported training algorithms: pa

    error_sensitive : bool, optional (default=True)
        If this parameter is True, the optimization routine includes
        into the objective function the square root of the number of
        incorrect labels predicted by the model.

        Supported training algorithms: pa

    averaging : bool, optional (default=True)
        If this parameter is True, the optimization routine computes
        the average of feature weights at all updates in the training
        process (similarly to Averaged Perceptron).

        Supported training algorithms: pa

    variance : float, optional (default=1)
        The initial variance of every feature weight.
        The algorithm initialize a vector of feature weights as
        a multivariate Gaussian distribution with mean 0
        and variance `variance`.

        Supported training algorithms: arow

    gamma : float, optional (default=1)
        The tradeoff between loss function and changes of feature weights.

        Supported training algorithms: arow

    verbose : bool, optional (default=False)
        Enable trainer verbose mode.

    model_filename : str, optional (default=None)
        A path to an existing CRFSuite model.
        This parameter allows to load and use existing crfsuite models.

        By default, model files are created automatically and saved
        in temporary locations; the preferred way to save/load CRF models
        is to use pickle (or its alternatives like joblib).

    """
    def __init__(self,
                 algorithm=None,

                 min_freq=None,
                 all_possible_states=None,
                 all_possible_transitions=None,
                 c1=None,
                 c2=None,
                 max_iterations=None,
                 num_memories=None,
                 epsilon=None,
                 period=None,
                 delta=None,
                 linesearch=None,
                 max_linesearch=None,
                 calibration_eta=None,
                 calibration_rate=None,
                 calibration_samples=None,
                 calibration_candidates=None,
                 calibration_max_trials=None,
                 pa_type=None,
                 c=None,
                 error_sensitive=None,
                 averaging=None,
                 variance=None,
                 gamma=None,

                 verbose=False,
                 model_filename=None,
                 keep_tempfiles=False,
                 trainer_cls=None):

        self.algorithm = algorithm
        self.min_freq = min_freq
        self.all_possible_states = all_possible_states
        self.all_possible_transitions = all_possible_transitions
        self.c1 = c1
        self.c2 = c2
        self.max_iterations = max_iterations
        self.num_memories = num_memories
        self.epsilon = epsilon
        self.period = period
        self.delta = delta
        self.linesearch = linesearch
        self.max_linesearch = max_linesearch
        self.calibration_eta = calibration_eta
        self.calibration_rate = calibration_rate
        self.calibration_samples = calibration_samples
        self.calibration_candidates = calibration_candidates
        self.calibration_max_trials = calibration_max_trials
        self.pa_type = pa_type
        self.c = c
        self.error_sensitive = error_sensitive
        self.averaging = averaging
        self.variance = variance
        self.gamma = gamma
        self.keep_tempfiles = keep_tempfiles
        self.model_filename = model_filename


        self.modelfile = FileResource(
            filename=model_filename,
            keep_tempfiles=keep_tempfiles,
            suffix=".crfsuite",
            prefix="model"
        )
        self.verbose = verbose
        self.trainer_cls = trainer_cls
        self.training_log_ = None

        self._tagger = None
        self._info_cached = None

    def fit(self, X, y, X_dev=None, y_dev=None):
        """
        Train a model.

        Parameters
        ----------
        X : list of lists of dicts
            Feature dicts for several documents (in a python-crfsuite format).

        y : list of lists of strings
            Labels for several documents.

        X_dev : (optional) list of lists of dicts
            Feature dicts used for testing.

        y_dev : (optional) list of lists of strings
            Labels corresponding to X_dev.
        """
        if (X_dev is None and y_dev is not None) or (X_dev is not None and y_dev is None):
            raise ValueError("Pass both X_dev and y_dev to use the holdout data")

        if self._tagger is not None:
            self._tagger.close()
            self._tagger = None
            self._info_cached = None
        self.modelfile.refresh()

        trainer = self._get_trainer()
        train_data = zip(X, y)

        if self.verbose:
            train_data = tqdm(train_data, "loading training data to CRFsuite", len(X), leave=True)

        for xseq, yseq in train_data:
            trainer.append(xseq, yseq)

        if self.verbose:
            print("")

        if X_dev is not None:
            test_data = zip(X_dev, y_dev)

            if self.verbose:
                test_data = tqdm(test_data, "loading dev data to CRFsuite", len(X_dev), leave=True)

            for xseq, yseq in test_data:
                trainer.append(xseq, yseq, 1)

            if self.verbose:
                print("")

        trainer.train(self.modelfile.name, holdout=-1 if X_dev is None else 1)
        self.training_log_ = trainer.logparser
        return self

    def predict(self, X):
        """
        Make a prediction.

        Parameters
        ----------
        X : list of lists of dicts
            feature dicts in python-crfsuite format

        Returns
        -------
        y : list of lists of strings
            predicted labels

        """
        return list(map(self.predict_single, X))

    def predict_single(self, xseq):
        """
        Make a prediction.

        Parameters
        ----------
        xseq : list of dicts
            feature dicts in python-crfsuite format

        Returns
        -------
        y : list of strings
            predicted labels

        """
        return self.tagger_.tag(xseq)

    def predict_marginals(self, X):
        """
        Make a prediction.

        Parameters
        ----------
        X : list of lists of dicts
            feature dicts in python-crfsuite format

        Returns
        -------
        y : list of lists of dicts
            predicted probabilities for each label at each position

        """
        return list(map(self.predict_marginals_single, X))

    def predict_marginals_single(self, xseq):
        """
        Make a prediction.

        Parameters
        ----------
        xseq : list of dicts
            feature dicts in python-crfsuite format

        Returns
        -------
        y : list of dicts
            predicted probabilities for each label at each position

        """
        labels = self.tagger_.labels()
        self.tagger_.set(xseq)
        return [
            {label: self.tagger_.marginal(label, i) for label in labels}
            for i in range(len(xseq))
        ]

    def score(self, X, y):
        """
        Return accuracy score computed for sequence items.

        For other metrics check :mod:`sklearn_crfsuite.metrics`.
        """
        from sklearn_crfsuite.metrics import flat_accuracy_score
        y_pred = self.predict(X)
        return flat_accuracy_score(y, y_pred)

    @property
    def tagger_(self):
        """
        pycrfsuite.Tagger instance.
        """
        if self._tagger is None:
            if self.modelfile.name is None:
                return None

            tagger = pycrfsuite.Tagger()
            tagger.open(self.modelfile.name)
            self._tagger = tagger
            self._info_cached = None
        return self._tagger

    @property
    def classes_(self):
        """
        A list of class labels.
        """
        if self.tagger_ is None:
            return None
        return self.tagger_.labels()

    @property
    def size_(self):
        """
        Size of the CRF model, in bytes.
        """
        if self._info is None:
            return None
        return int(self._info.header['size'])

    @property
    def num_attributes_(self):
        """
        Number of non-zero CRF attributes.
        """
        if self._info is None:
            return None
        return int(self._info.header['num_attrs'])

    @property
    def attributes_(self):
        """
        A list of known attributes.
        """
        if self._info is None:
            return None

        attrs = [None for _ in range(self.num_attributes_)]
        for name, value in self._info.attributes.items():
            attrs[int(value)] = name

        return attrs

    @property
    def state_features_(self):
        """
        Dict with state feature coefficients:
        ``{(attr_name, label): coef}``
        """
        if self._info is None:
            return None
        return self._info.state_features

    @property
    def transition_features_(self):
        """
        Dict with transition feature coefficients:
        ``{(label_from, label_to): coef}``
        """
        if self._info is None:
            return None
        return self._info.transitions

    @property
    def _info(self):
        if self.tagger_ is None:
            return None
        if self._info_cached is None:
            self._info_cached = self.tagger_.info()
        return self._info_cached

    def _get_trainer(self):
        trainer_cls = self.trainer_cls or LinePerIterationTrainer
        params = {
            'feature.minfreq': self.min_freq,
            'feature.possible_states': self.all_possible_states,
            'feature.possible_transitions': self.all_possible_transitions,
            'c1': self.c1,
            'c2': self.c2,
            'max_iterations': self.max_iterations,
            'num_memories': self.num_memories,
            'epsilon': self.epsilon,
            'period': self.period,
            'delta': self.delta,
            'linesearch': self.linesearch,
            'max_linesearch': self.max_linesearch,
            'calibration.eta': self.calibration_eta,
            'calibration.rate': self.calibration_rate,
            'calibration.samples': self.calibration_samples,
            'calibration.candidates': self.calibration_candidates,
            'calibration.max_trials': self.calibration_max_trials,
            'type': self.pa_type,
            'c': self.c,
            'error_sensitive': self.error_sensitive,
            'averaging': self.averaging,
            'variance': self.variance,
            'gamma': self.gamma,
        }
        params = {k: v for k, v in params.items() if v is not None}
        return trainer_cls(
            algorithm=self.algorithm,
            params=params,
            verbose=self.verbose,
        )

    def __getstate__(self):
        dct = self.__dict__.copy()
        dct['_tagger'] = None
        dct['_info_cached'] = None
        return dct

In [122]:
from nltk.tokenize import word_tokenize



all_text_c['text'] = all_text['text'].apply(word_tokenize)



In [123]:
def add_to_arr(splitted):

  if splitted:
    len(splitted[0] > 0)


def split_sen(d, spl_by):
  res = []
  for w in d:
    l = re.split(spl_by, w)
    if l:
      for wl in l:
        if len(wl) > 0:
          res.append(wl)
    else:
      res.append(w)
  return res

all_text_t['text'] = all_text_c['text'].apply(lambda d:  split_sen(d, '(?!\w+)\/(?=\w+)'))
all_text_t['text'] = all_text_t['text'].apply(lambda d: split_sen(d, '[A-Z]{3,}(?=\w+)'))

In [124]:
from nltk.corpus import wordnet
import spacy
from nltk.stem.snowball import SnowballStemmer

import en_core_web_sm
nlp = en_core_web_sm.load()
#nlp = spacy.load("en_core_web_sm")

stemmer = SnowballStemmer("english")


fin_word = {"a",
"about",
"above",
"after",
"again",
"against",
"all",
"am",
"an",
"and",
"any",
"are",
"aren't",
"as",
"at",
"be",
"because",
"been",
"before",
"being",
"below",
"between",
"both",
"but",
"by",
"can't",
"cannot",
"could",
"couldn't",
"did",
"didn't",
"do",
"does",
"doesn't",
"doing",
"don't",
"down",
"during",
"each",
"few",
"for",
"from",
"further",
"had",
"hadn't",
"has",
"hasn't",
"have",
"haven't",
"having",
"he",
"he'd",
"he'll",
"he's",
"her",
"here",
"here's",
"hers",
"herself",
"him",
"himself",
"his",
"how",
"how's",
"i",
"i'd",
"i'll",
"i'm",
"i've",
"if",
"in",
"into",
"is",
"isn't",
"it",
"it's",
"its",
"itself",
"let's",
"me",
"more",
"most",
"mustn't",
"my",
"myself",
"no",
"nor",
"not",
"of",
"off",
"on",
"once",
"only",
"or",
"other",
"ought",
"our",
"ours 	ourselves",
"out",
"over",
"own",
"same",
"shan't",
"she",
"she'd",
"she'll",
"she's",
"should",
"shouldn't",
"so",
"some",
"such",
"than",
"that",
"that's",
"the",
"their",
"theirs",
"them",
"themselves",
"then",
"there",
"there's",
"these",
"they",
"they'd",
"they'll",
"they're",
"they've",
"this",
"those",
"through",
"to",
"too",
"under",
"until",
"up",
"very",
"was",
"wasn't",
"we",
"we'd",
"we'll",
"we're",
"we've",
"were",
"weren't",
"what",
"what's",
"when",
"when's",
"where",
"where's",
"which",
"while",
"who",
"who's",
"whom",
"why",
"why's",
"with",
"won't",
"would",
"wouldn't",
"you",
"you'd",
"you'll",
"you're",
"you've",
"your",
"yours",
"yourself",
"yourselves",
"“",
",",
"”",
"'s"
"."}

def rem_stopwords(sent):


  # Remove stopwords
  filtered_words = []

  for token in sent:

    if not token.lower() in fin_word:
      if not token.isupper():
        if not token in string.punctuation:
          filtered_words.append(token)


  # Join the filtered words to form a clean text
  return filtered_words

all_text_temp = all_text_t['text'].apply(rem_stopwords)


In [167]:
lemmatizer = WordNetLemmatizer()

# function to convert nltk tag to wordnet tag
def nltk_tag_to_wordnet_tag(nltk_tag):
    if nltk_tag.startswith('J'):
        return wordnet.ADJ
    elif nltk_tag.startswith('V'):
        return wordnet.VERB
    elif nltk_tag.startswith('N'):
        return wordnet.NOUN
    elif nltk_tag.startswith('R'):
        return wordnet.ADV
    else:
        return None

def lemmatize_sentence(sentence_tokenized):
    #tokenize the sentence and find the POS tag for each token
    nltk_tagged = nltk.pos_tag(sentence_tokenized)
    #tuple of (token, wordnet_tag)
    wordnet_tagged = map(lambda x: (x[0], nltk_tag_to_wordnet_tag(x[1])), nltk_tagged)
    lemmatized_sentence = []
    for i, word_tag in enumerate(wordnet_tagged):
        word,tag = word_tag
        if tag is None:
            #if there is no available tag, append the token as is
            lemmatized_sentence.append((word, "UNK"))
        else:
            #else use the tag to lemmatize the token
            lemmatized_sentence.append((lemmatizer.lemmatize(word, tag), nltk_tagged[i][1]))
    return lemmatized_sentence
all_text_temp2 = []
for i,tokens in enumerate(all_text_temp):
  all_text_temp2.append([])
  for word in tokens:
    if len(word) > 2:
      all_text_temp2[i].append(word)

all_text_t['text'] = all_text_temp2
all_text_t['text'] =  all_text_t['text'].apply(lemmatize_sentence)

In [168]:
all_text_t = all_text_t.sample(frac=1).reset_index(drop=True)

all_text_t[:4]

,text,true_sentiment
0,"[(steady, NNS), (find, VBG), (support, NN), (l...",Negative
1,"[(cheer, NNS), (broad, JJ), (Dollar, NNP), (we...",Negative
2,"[(stay, NNS), (bear, VBP), (radar, NN), (even,...",Positive
3,"[(Sterling, NN), (seem, VBZ), (take, VBG), (li...",Positive


In [128]:
#txt_sent = "/content/drive/MyDrive/HLT project/Project files/NRC-VAD-Lexicon.txt"
txt_sent = "/home/lesi-nedo/Desktop/master/second-semester/HLT/project/NRC-VAD-Lexicon.txt"

data = {}

with open(txt_sent, "r") as f:
  t = f.read()
  for si in t.split('\n'):
    si_w = (si +'\t').split('\t')
    if si_w[0] != '':
      data[si_w[0]] = list(map(lambda fl: float(fl), si_w[1:-1]))
    else:
      pass

In [169]:


#txt_sent = "/content/drive/MyDrive/HLT project/Project files/SentiWordNet_3.0.0.txt"
txt_sent = "/home/lesi-nedo/Desktop/master/second-semester/HLT/project/SentiWordNet_3.0.0.txt"
senti_word = {}
regex = '0(\.\d+)?\s0(\.\d+)?\s([a-z_\-]+#\d+\s)+'
pattern = re.compile(regex)


with open(txt_sent, "r") as f:
  t = f.read()
  for si in t.split("\n"):
    if len(si) > 0 and si[0] != '#':
      matched = pattern.search(si)
      if matched:
        val = matched.group(0).split('\t')
        scor = [float(val[0]), float(val[1])]
        wors = re.split('#\d+\s?', val[2])
        wors.pop(-1)
        for wr in wors:
          senti_word[wr] = scor

df_senti = pd.DataFrame(senti_word, index=["pos_score", "negative_score"])




In [170]:

for w,v in data.items():
  if len(v) < 3:
    print(w)
df2 = pd.DataFrame(data, index=['valence', 'arousal', 'dominance'])


In [171]:
def positional_encoding(length, depth):
  depth = depth/2

  positions = np.arange(length)[:, np.newaxis]     # (seq, 1)
  depths = np.arange(depth)[np.newaxis, :]/depth   # (1, depth)

  angle_rates = 1 / (10000**depths)         # (1, depth)
  angle_rads = positions * angle_rates      # (pos, depth)

  pos_encoding = np.concatenate(
      [np.sin(angle_rads), np.cos(angle_rads)],
      axis=-1)

  return pos_encoding.astype(dtype=np.float32)

pos_embeddings = []
for i,sent in enumerate(all_text_t['text']):
  pos_embeddings.append(positional_encoding(len(sent), 256))


In [231]:
all_tfidf_mat = []
def identity_tokenizer(text):
  return text[0].lower()

tfidf = TfidfVectorizer(tokenizer=identity_tokenizer, lowercase=False)




size_data = all_text_t['text'].size

for i in range(size_data):
  tfidf_vals = tfidf.fit_transform(all_text_t['text'].iloc[i])
  all_tfidf_mat.append(tfidf_vals)



/home/lesi-nedo/miniforge3/envs/HLT/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [272]:
all_tfidf_mat[i]

<168x32 sparse matrix of type '<class 'numpy.float64'>'
	with 917 stored elements in Compressed Sparse Row format>

In [273]:
import math

ng_random = np.random.RandomState(89377)

def find_synonyms(keyword) :

    synonyms = []
    for synset in wordnet.synsets(keyword):
        for lemma in synset.lemmas():
            synonyms.append(lemma.name())

    return str(synonyms)

def sigmoid(x):
  return 1 / (1 + math.exp(-x))

def word2features(sent, num_sent, i):

    word = sent[i][0]
    postag = sent[i][1]
    word_low = word.lower()
    regex= "^(\+|-)\d+\.\d+"
    pattern = re.compile(regex)




    features = {
        'bias':1.0,
        'word.lower()': word_low,
        'len(word)': len(word),
        'word.isdigit()': word.isdigit(),
        'postag': postag,
        'postag[:2]': postag[:2],
        'word.stemmed': re.sub(r'(.{2,}?)([aeiougyn]+$)',r'\1', word_low),
        "pos_encoding_sum": sigmoid(np.sum(pos_embeddings[num_sent][i])),
        "synon_word": find_synonyms(word_low)[1],
        "tf_idf_mean": np.sum(all_tfidf_mat[num_sent][i]),

    }


    try:
      features.update({
        "def_word": wordnet.synsets(word_low)[0].definition(),
      })
    except IndexError:
      features.update({
        "def_word":0,
      })

    try:
      sc = df_senti[word_low]
      sentim =  df2[word_low]
      features.update({
              'word.dff_score_single': sc[0]-sc[1],
              'word.val_score': sigmoid(sentim.sum())
                              + sc[0] + sc[1]
          })
    except KeyError:
      features.update({
              'word.dff_score_single': 0,
              'word.val_score': 0
          })
    obj_mat = pattern.match(word)
    if pattern.match(word) is not None:
      features['interests'] = word

    if i > 0:
        word1 = sent[i-1][0]
        postag1 = sent[i-1][1]


        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:postag': postag1,
            '-1:postag[:2]': postag1[:2],
            "-1:pos_encoding_sum": sigmoid(np.sum(pos_embeddings[num_sent][i-1])),
            "-1:synon_word": find_synonyms(word1.lower())[1],
            "-1:tf_idf_mean": np.sum(all_tfidf_mat[num_sent][i-1])
        })
        com_wrod = word1.lower()+ '_' + word_low
        try:
          sc = df_senti[com_wrod]
          features.update({
              '-1:word.score_double_prev': sc[0] - sc[1],
              '-1:word.comp': com_wrod,
          })
        except KeyError:
          features.update({
                '-1:word.score_double_prev': 0,
                '-1:word.comp': com_wrod,
            })
          pass
    else:
        features['BOS'] = True

    if i < len(sent)-1:
        word1 = sent[i+1][0]
        postag1 = sent[i+1][1]

        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:postag': postag1,
            '+1:postag[:2]': postag1[:2],
            "+1:pos_encoding_sum": sigmoid(np.sum(pos_embeddings[num_sent][i+1])),
            "+1:synon_word": find_synonyms(word1.lower())[1],
            "+1:tf_idf_mean": np.sum(all_tfidf_mat[num_sent][i+1])
        })

        com_wrod = word1.lower()+ '_' + word_low
        try:
          sc = df_senti[com_wrod]
          features.update({
              '+1:word.score_double_next': sc[0]-sc[1],
              '+1:word.comp': com_wrod,
          })
        except KeyError:
          features.update({
              '+1:word.score_double_next': 0,
              '+1:word.comp': com_wrod,
          })
    else:
        features['EOS'] = True

    try:
      sum_val = df2[word_low].sum()
      features.update({
          "sig_val": sigmoid(sum_val)
      })

    except KeyError:
      features.update({
          "sig_val":0
      })

    return features


def sent2features(sent, num_sent):
    return [word2features(sent, num_sent, i) for i in range(len(sent))]


def sent2labels(sent):
    return [label for token, postag, label in sent]

def sent2tokens(sent):
    return [token for token, postag, label in sent]

In [274]:

all_X = [sent2features(s, i) for i,s in enumerate(all_text_t['text'])]


/tmp/ipykernel_487204/1722336413.py:56: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'word.dff_score_single': sc[0]-sc[1],
/tmp/ipykernel_487204/1722336413.py:58: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  + sc[0] + sc[1]
/tmp/ipykernel_487204/1722336413.py:86: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  '-1:word.score_double_prev': sc[0] - sc[1],
/tmp/ipykernel_487204/1722336413.py:115: FutureWarning: Series.__getitem__ treat

In [275]:


all_Y = [list(np.repeat(l, len(all_X[i]))) for i,l in enumerate(all_text_t['true_sentiment'])]

In [276]:
all_l = len(all_X)

train_ind = int(all_l * 0.8)
X_train = all_X[0:train_ind]
X_test = all_X[train_ind:all_l]

y_train = all_Y[0:train_ind]

y_test = all_Y[train_ind:all_l]



In [280]:
%%time
from sklearn.model_selection import cross_val_score

crf = CRF(
    algorithm='lbfgs',
    c1=0.4,
    c2=0.016,
    max_iterations=400,
    all_possible_transitions=True,
    all_possible_states=True,
    verbose=True,
    num_memories=16,
    linesearch='Backtracking',
    period=1000,
    delta=1e-9,
    max_linesearch=40,

)

crf.fit(X_train, y_train)

loading training data to CRFsuite: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1832/1832 [00:03<00:00, 575.82it/s]



Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 1
feature.possible_transitions: 1
0....1....2....3....4....5....6....7....8....9....10
Number of features: 890510
Seconds required: 1.837

L-BFGS optimization
c1: 0.400000
c2: 0.016000
num_memories: 16
max_iterations: 400
epsilon: 0.000010
stop: 1000
delta: 0.000000
linesearch: Backtracking
linesearch.max_iterations: 40

Iter 1   time=0.80  loss=301000.50 active=389101 feature_norm=0.12
Iter 2   time=0.18  loss=290303.11 active=548499 feature_norm=0.16
Iter 3   time=0.19  loss=230443.57 active=543975 feature_norm=0.63
Iter 4   time=0.19  loss=67598.99 active=542582 feature_norm=2.85
Iter 5   time=1.01  loss=67438.34 active=540864 feature_norm=2.92
Iter 6   time=0.35  loss=55739.54 active=557023 feature_norm=3.42
Iter 7   time=0.55  loss=53744.28 active=570350 feature_norm=3.64
Iter 8   time=0.36  loss=47281.36 active=542107 feature_norm=3.90
Iter 9   time=0.35  loss=36389.66 active=539006 feature_norm=4.

CRF(algorithm='lbfgs', all_possible_states=True, all_possible_transitions=True,
    c1=0.4, c2=0.016, delta=1e-09, linesearch='Backtracking',
    max_iterations=400, max_linesearch=40, num_memories=16, period=1000,
    verbose=True)

In [238]:
labels = list(crf.classes_)
labels


['Negative', 'Positive', 'Neutral']

In [282]:
y_pred = crf.predict(X_test)
metrics.flat_f1_score(y_test, y_pred,
                      average='weighted', labels=labels)


0.38778445429761854

In [145]:
metrics.flat_f1_score(y_test, y_pred,
                      average='weighted', labels=labels)
print('model size: {:0.2f}M'.format(crf.size_ / 1000000))

model size: 3.62M


In [31]:
from sklearn_crfsuite.utils import flatten
from sklearn import metrics as skmetrics

sorted_labels = sorted(
    labels,
    key=lambda name: (name[1:], name[0])
)
y_test_flat = flatten(y_test)
y_pred_flat = flatten(y_pred)
print(skmetrics.classification_report(y_test_flat, y_pred_flat, labels=sorted_labels, digits=3))

              precision    recall  f1-score   support

    Negative      0.272     0.347     0.305     25857
     Neutral      0.352     0.326     0.339     37516
    Positive      0.336     0.293     0.313     33608

    accuracy                          0.320     96981
   macro avg      0.320     0.322     0.319     96981
weighted avg      0.325     0.320     0.321     96981



In [ ]:
%%time

params_space = {
    'c1': scipy.stats.expon(scale=0.5),
    'c2': scipy.stats.expon(scale=0.05),
}

# use the same metric for evaluation
f1_scorer = make_scorer(metrics.flat_f1_score,
                        average='weighted', labels=labels)

# search
rs = RandomizedSearchCV(crf, params_space,
                        cv=3,
                        verbose=3,
                        n_jobs=11,
                        n_iter=5,
                        scoring=f1_scorer)
rs.fit(X_train, y_train)

Fitting 3 folds for each of 5 candidates, totalling 15 fits
CPU times: user 3min 13s, sys: 4.92 s, total: 3min 18s
Wall time: 10min 27s


RandomizedSearchCV(cv=3,
                   estimator=CRF(algorithm='lbfgs', all_possible_states=True,
                                 all_possible_transitions=True,
                                 c1=0.014662588440616717,
                                 c2=0.27612818382193377, delta=1e-09,
                                 linesearch='Backtracking', max_iterations=500,
                                 max_linesearch=40, num_memories=16,
                                 period=1000),
                   n_iter=5, n_jobs=11,
                   param_distributions={'c1': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x78b220b58110>,
                                        'c2': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x78b2201b74d0>},
                   scoring=make_scorer(flat_f1_score, response_method='predict', average=weighted, labels=['Neutral', 'Negative', 'Positive']),
                   verbose=3)

In [ ]:
crf = rs.best_estimator_
y_pred = crf.predict(X_test)
y_test_flat = flatten(y_test)
y_pred_flat = flatten(y_pred)
print(skmetrics.classification_report(y_test_flat, y_pred_flat, labels=sorted_labels, digits=3))

              precision    recall  f1-score   support

    Negative      0.574     0.592     0.583     23681
     Neutral      0.488     0.532     0.509     22402
    Positive      0.565     0.494     0.528     22396

    accuracy                          0.541     68479
   macro avg      0.542     0.540     0.540     68479
weighted avg      0.543     0.541     0.541     68479



In [ ]:
def print_state_features(state_features):
    for (attr, label), weight in state_features:
        print("%0.6f %-8s %s" % (weight, label, attr))

print("Top positive:")
print_state_features(Counter(crf.state_features_).most_common(30))

print("\nTop negative:")
print_state_features(Counter(crf.state_features_).most_common()[-30:])